# Customer Churn Prediction

This notebook builds a simple stacking ensemble for the Playground Series Season 6 Episode 3 churn task.

In [ ]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:   
        print(os.path.join(dirname, filename))


import warnings
warnings.filterwarnings("ignore")

/kaggle/input/competitions/playground-series-s6e3/sample_submission.csv
/kaggle/input/competitions/playground-series-s6e3/train.csv
/kaggle/input/competitions/playground-series-s6e3/test.csv


In [2]:
train_path = "/kaggle/input/competitions/playground-series-s6e3/train.csv"

## Load the training data

Read the training file and prepare it for exploration.

In [ ]:
train_df = pd.read_csv(train_path, engine='pyarrow', dtype_backend='numpy_nullable')
train_df = train_df.drop(columns=['id'])
train_df.head()

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,Male,0,Yes,Yes,29,Yes,No,DSL,Yes,No,Yes,Yes,No,No,One year,Yes,Mailed check,60.1,1653.85,No
1,Male,0,Yes,Yes,58,Yes,No,DSL,Yes,Yes,No,Yes,Yes,No,Two year,No,Credit card (automatic),69.5,3778.2,No
2,Male,0,Yes,No,58,Yes,Yes,Fiber optic,No,Yes,No,No,Yes,Yes,Month-to-month,Yes,Electronic check,100.4,5841.35,No
3,Female,0,No,No,1,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,69.7,70.7,Yes
4,Female,0,No,No,1,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.45,70.45,Yes


In [4]:
train_df.shape

(594194, 20)

## Split features and target

Separate the target column and create the train test split.

In [ ]:
from sklearn.model_selection import train_test_split

cat_cols = ['gender','Partner','Dependents','PhoneService','MultipleLines',
            'InternetService','OnlineSecurity','OnlineBackup','DeviceProtection',
            'TechSupport','StreamingTV','StreamingMovies','Contract',
            'PaperlessBilling','PaymentMethod']
num_cols = ['SeniorCitizen','tenure','MonthlyCharges','TotalCharges']

X = train_df.drop(columns=['Churn'])
y = (train_df['Churn'] == 'Yes').astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

## Load the test data

Read the test file that will be used for final predictions.

In [ ]:
test_path = "/kaggle/input/competitions/playground-series-s6e3/test.csv"
test_df = pd.read_csv(test_path, engine='pyarrow', dtype_backend='numpy_nullable')

## Set up preprocessing

Define the encoders and scaler for the model pipelines.

In [ ]:
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder
from sklearn.compose import ColumnTransformer

In [ ]:
lr_preprocessor = ColumnTransformer([
    ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_cols),
    ('num', StandardScaler(), num_cols)
])

In [ ]:
tree_preprocessor = ColumnTransformer([
    ('cat', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1), cat_cols),
    ('num', 'passthrough', num_cols)
])

## Build the models

Create the model pipelines and combine them with stacking.

In [10]:
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier, StackingClassifier, ExtraTreesClassifier
from sklearn.linear_model import LogisticRegression
from lightgbm import LGBMClassifier

In [11]:
lgbm_pipe = Pipeline([
    ('pre', tree_preprocessor),
    ('model', LGBMClassifier(
        random_state=42,
        class_weight='balanced',
        verbose=-1
    ))
])

lr_pipe = Pipeline([
    ('pre', lr_preprocessor),
    ('model', LogisticRegression(
        max_iter=1000,
        class_weight='balanced'))
])

et_pipe = Pipeline([
    ('pre', tree_preprocessor),
    ('model', ExtraTreesClassifier(
        n_estimators=100,
        n_jobs=-1,
        random_state=42,
        class_weight='balanced'
    ))
])


In [12]:
stacking_clf = StackingClassifier(
    estimators=[
        ('lgbm', lgbm_pipe),
        ('et',   et_pipe),
        ('lr',   lr_pipe)
    ],
    final_estimator=LogisticRegression(),
    cv=3,
    stack_method='predict_proba'
)

## Train and evaluate

Fit the ensemble and check the validation report.

In [13]:
stacking_clf.fit(X_train, y_train)

StackingClassifier(cv=3,
                   estimators=[('lgbm',
                                Pipeline(steps=[('pre',
                                                 ColumnTransformer(transformers=[('cat',
                                                                                  OrdinalEncoder(),
                                                                                  ['gender',
                                                                                   'Partner',
                                                                                   'Dependents',
                                                                                   'PhoneService',
                                                                                   'MultipleLines',
                                                                                   'InternetService',
                                                                                   'OnlineSecurity',
                                                                                   'OnlineBackup',
                                                                                   'DeviceProtection',
                                                                                   'TechSupport',
                                                                                   'StreamingTV',
                                                                                   'StreamingMovies',
                                                                                   'Contract',
                                                                                   'PaperlessBilling',
                                                                                   'PaymentMethod']),...
                                                                                   'OnlineBackup',
                                                                                   'DeviceProtection',
                                                                                   'TechSupport',
                                                                                   'StreamingTV',
                                                                                   'StreamingMovies',
                                                                                   'Contract',
                                                                                   'PaperlessBilling',
                                                                                   'PaymentMethod']),
                                                                                 ('num',
                                                                                  StandardScaler(),
                                                                                  ['SeniorCitizen',
                                                                                   'tenure',
                                                                                   'MonthlyCharges',
                                                                                   'TotalCharges'])])),
                                                ('model',
                                                 LogisticRegression(class_weight='balanced',
                                                                    max_iter=1000))]))],
                   final_estimator=LogisticRegression(),
                   stack_method='predict_proba')

In [14]:
from sklearn.metrics import classification_report
y_pred2 = stacking_clf.predict(X_test)
print(classification_report(y_test, y_pred2))

              precision    recall  f1-score   support

          No       0.91      0.90      0.91     91935
         Yes       0.68      0.70      0.69     26904

    accuracy                           0.86    118839
   macro avg       0.80      0.80      0.80    118839
weighted avg       0.86      0.86      0.86    118839



## Create the submission

Predict the test set probabilities and save the submission file.

In [15]:
X_submit = test_df.drop(columns=['id'])

y_prob = stacking_clf.predict_proba(X_submit)[:, 1]

submission = pd.DataFrame({
    'id': test_df['id'],
    'Churn': y_prob
})
submission.to_csv('submission.csv', index=False)